Imports & Load Preprocessed Data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)


In [3]:
df = pd.read_csv("../data/processed/incident_preprocessed.csv")
df.shape

(24918, 27)

Create aging_bucket (CORE FEATURE)

In [4]:
df['aging_bucket'] = pd.cut(
    df['aging_days'],
    bins=[-1, 1, 7, 30, np.inf],
    labels=['0–1 days', '2–7 days', '8–30 days', '30+ days']
)

In [5]:
df['aging_bucket'].value_counts()

aging_bucket
0–1 days     13918
2–7 days      6029
8–30 days     3828
30+ days      1143
Name: count, dtype: int64

Temporal Features from opened_at

In [6]:
df['opened_at'] = pd.to_datetime(df['opened_at'])

df['hour_of_day'] = df['opened_at'].dt.hour
df['day_of_week'] = df['opened_at'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)


Encode Severity Signals (Priority / Impact / Urgency)

In [7]:
severity_cols = ['priority', 'impact', 'urgency']

for col in severity_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

Handle High-Cardinality location (Smartly)

One-hot encoding location would explode features.

We use frequency encoding instead.

In [8]:
location_freq = df['location'].value_counts(normalize=True)
df['location_freq'] = df['location'].map(location_freq)

In [9]:
df = df.drop(columns=['location'])


Drop Columns No Longer Needed

In [10]:
df = df.drop(columns=['opened_at'])

Final Feature Sanity Check

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24918 entries, 0 to 24917
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   sys_mod_count            24918 non-null  int64   
 1   made_sla                 24918 non-null  bool    
 2   caller_id                24918 non-null  object  
 3   opened_by                24918 non-null  object  
 4   contact_type             24918 non-null  object  
 5   category                 24918 non-null  object  
 6   subcategory              24918 non-null  object  
 7   u_symptom                24918 non-null  object  
 8   impact                   0 non-null      float64 
 9   urgency                  0 non-null      float64 
 10  assignment_group         24918 non-null  object  
 11  assigned_to              24918 non-null  object  
 12  knowledge                24918 non-null  bool    
 13  u_priority_confirmation  24918 non-null  bool    
 14  notify

In [12]:
df.head()

,sys_mod_count,made_sla,caller_id,opened_by,contact_type,category,subcategory,u_symptom,impact,urgency,assignment_group,assigned_to,knowledge,u_priority_confirmation,notify,problem_id,rfc,caused_by,closed_code,resolved_by,resolution_time_hours,aging_days,priority,sla_breach_flag,created_month,aging_bucket,hour_of_day,day_of_week,is_weekend,location_freq
0,0,True,Caller 2403,Opened by 8,Phone,Category 55,Subcategory 170,Symptom 72,NaN,NaN,Group 56,?,True,False,Do Not Notify,?,?,?,code 5,Resolved by 149,10.216667,0,3,0,2016-02,0–1 days,1,0,0,0.131190
1,0,True,Caller 2403,Opened by 397,Phone,Category 40,Subcategory 215,Symptom 471,NaN,NaN,Group 70,Resolver 89,True,False,Do Not Notify,?,?,?,code 5,Resolved by 81,29.200000,1,3,1,2016-02,0–1 days,4,0,0,0.002769
2,0,True,Caller 4416,Opened by 8,Phone,Category 20,Subcategory 125,Symptom 471,NaN,NaN,Group 70,?,True,False,Do Not Notify,?,?,?,code 10,Resolved by 5,20.750000,0,3,0,2016-02,0–1 days,6,0,0,0.223092
3,0,True,Caller 4491,Opened by 180,Phone,Category 9,Subcategory 97,Symptom 450,NaN,NaN,Group 25,Resolver 125,True,False,Do Not Notify,?,?,?,code 3,Resolved by 113,53.466667,2,3,1,2016-02,2–7 days,6,0,0,0.223092
4,0,True,Caller 3765,Opened by 180,Phone,Category 53,Subcategory 168,Symptom 232,NaN,NaN,Group 70,?,True,False,Do Not Notify,?,?,?,code 7,Resolved by 62,8.883333,0,3,0,2016-02,0–1 days,6,0,0,0.077895


Save Feature Dataset

In [13]:
df.to_csv(
    "../data/processed/incident_ml_features.csv",
    index=False
)

### Notebook 03 Summary

- Engineered business-aligned features from raw incident data
- Created aging buckets from aging_days
- Derived temporal features from incident creation time
- Preserved ordinal severity signals
- Applied frequency encoding for high-cardinality location
- Removed redundant raw columns
- Produced a clean, explainable feature dataset ready for modeling